# V2 — 101× stocks: drawdown, matched controls and post-trough divergence

Standalone scan. It does **not** use V5 entry signals.

NSE data is stored as **one parquet per trading day** (`nse_cm_YYYYMMDD.parquet`), with every listed name in that file. A previous version scanned each file in isolation, so every symbol had **one bar** and nothing could pass the length filter. This notebook **stitches the full history per symbol** first.

**10,000% return** means close reaches **101×** entry close.

**Path:** buy at the expanding-min close; stop at the first later close ≥ 101× that low. Drawdown is only between those two dates.


V2 adds matched-control cohorts and forward recovery/landmark analysis. Original cells retained. Run top-to-bottom.


In [ ]:
from pathlib import Path
import warnings, gc
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from quant.config import DATA_DIR as QUANT_DATA_DIR, PARQUET_DATA_DIR, NIFTY50_PATH

DATA_DIR = PARQUET_DATA_DIR
OUT_DIR = QUANT_DATA_DIR / "results" / "hundred_baggers"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PANEL_PATH = OUT_DIR / "_panel_ohlc.parquet"
FEATURES_PATH = QUANT_DATA_DIR / "results" / "5x_turnaround_v5" / "features" / "stock_features.parquet"

MIN_RETURN = 100.0
TARGET_MULT = 1.0 + MIN_RETURN
MIN_BARS = 60
MIN_ENTRY_PRICE = 1.0

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 80)
pd.options.mode.copy_on_write = True

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print(f"Target: +{MIN_RETURN:.0f}%  ({TARGET_MULT:.0f}x)")
print("NIFTY:", NIFTY50_PATH, "exists" if NIFTY50_PATH.exists() else "missing")
assert DATA_DIR.exists(), f"Missing parquet dir: {DATA_DIR}"


In [ ]:
def first_series(frame, name):
    s = frame[name]
    return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s

def normalize_ohlc(df, source_name):
    cols = {str(c).lower().strip(): c for c in df.columns}
    aliases = {
        "date": ["date", "datetime", "timestamp"],
        "symbol": ["symbol", "ticker", "security", "stock"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close"],
        "volume": ["volume", "vol", "qty", "quantity", "total traded quantity"],
        "open": ["open"],
    }
    out = {}
    for target, names in aliases.items():
        found = next((cols[n] for n in names if n in cols), None)
        if found is not None:
            out[target] = found
    missing = [x for x in ["date", "high", "low", "close"] if x not in out]
    if missing:
        raise ValueError(f"{source_name}: missing {missing}; columns={list(df.columns)}")
    symbol = first_series(df, out["symbol"]) if "symbol" in out else Path(source_name).stem
    vol = pd.to_numeric(first_series(df, out["volume"]), errors="coerce") if "volume" in out else np.nan
    opn = pd.to_numeric(first_series(df, out["open"]), errors="coerce") if "open" in out else np.nan
    x = pd.DataFrame({
        "Date": pd.to_datetime(first_series(df, out["date"]), errors="coerce"),
        "Symbol": pd.Series(symbol, index=df.index).astype(str).str.upper().str.strip(),
        "Open": opn,
        "High": pd.to_numeric(first_series(df, out["high"]), errors="coerce"),
        "Low": pd.to_numeric(first_series(df, out["low"]), errors="coerce"),
        "Close": pd.to_numeric(first_series(df, out["close"]), errors="coerce"),
        "Volume": vol,
    })
    x = x.dropna(subset=["Date", "Close"])
    if getattr(x["Date"].dt, "tz", None) is not None:
        x["Date"] = x["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    x = x.loc[x["Close"] > 0]
    for c in ["Open", "High", "Low", "Close", "Volume"]:
        x[c] = x[c].astype("float32")
    x["Symbol"] = x["Symbol"].astype("string")
    return x

PANEL_SCHEMA = pa.schema([
    pa.field("Date", pa.timestamp("ns"), nullable=True),
    pa.field("Symbol", pa.large_string(), nullable=True),
    pa.field("Open", pa.float32(), nullable=True),
    pa.field("High", pa.float32(), nullable=True),
    pa.field("Low", pa.float32(), nullable=True),
    pa.field("Close", pa.float32(), nullable=True),
    pa.field("Volume", pa.float32(), nullable=True),
])

def max_dd_on_path(close, high, low):
    c = np.asarray(close, dtype=np.float64)
    h = np.asarray(high, dtype=np.float64)
    lo = np.asarray(low, dtype=np.float64)
    peak_c = np.maximum.accumulate(c)
    dd_c = float(np.nanmin(c / np.maximum(peak_c, 1e-12) - 1.0))
    peak_hl = np.maximum.accumulate(np.fmax(c, np.where(np.isfinite(h), h, c)))
    trough = np.where(np.isfinite(lo), lo, c)
    rel = trough / np.maximum(peak_hl, 1e-12) - 1.0
    dd_l = float(np.nanmin(rel))
    worst_i = int(np.nanargmin(rel))
    return dd_c, dd_l, worst_i

def first_target_run(dates, close, high, low, symbol):
    c = np.asarray(close, dtype=np.float64)
    n = len(c)
    if n < MIN_BARS:
        return None, 0.0, n
    rmin = np.inf
    rmin_i = -1
    max_mult = 0.0
    hit = None
    for i in range(n):
        px = c[i]
        if not np.isfinite(px) or px <= 0:
            continue
        if px < rmin:
            rmin = px
            rmin_i = i
        if rmin_i >= 0 and rmin > 0:
            max_mult = max(max_mult, px / rmin)
        if hit is None and rmin_i >= 0 and px >= rmin * TARGET_MULT:
            sl = slice(rmin_i, i + 1)
            dd_c, dd_l, w_i = max_dd_on_path(c[sl], high[sl], low[sl])
            d0 = pd.Timestamp(dates[rmin_i])
            d1 = pd.Timestamp(dates[i])
            hit = {
                "Symbol": symbol,
                "EntryDate": d0,
                "HitDate": d1,
                "EntryClose": float(rmin),
                "HitClose": float(px),
                "Return": float(px / rmin - 1.0),
                "ReturnPct": float((px / rmin - 1.0) * 100.0),
                "Multiple": float(px / rmin),
                "TradingDays": int(i - rmin_i),
                "CalendarDays": int((d1 - d0).days),
                "Years": float((d1 - d0).days / 365.25),
                "MaxDrawdownClose": dd_c,
                "MaxDrawdownLow": dd_l,
                "WorstDDDate": pd.Timestamp(dates[rmin_i + w_i]),
                "BarsInSample": n,
                "SampleStart": pd.Timestamp(dates[0]),
                "SampleEnd": pd.Timestamp(dates[-1]),
                "MaxMultipleFromExpandingMin": float(max_mult),
                "LowEntryFlag": bool(rmin < MIN_ENTRY_PRICE),
                "EntryIndex": int(rmin_i),
            }
    if hit is not None:
        hit["MaxMultipleFromExpandingMin"] = float(max_mult)
    return hit, float(max_mult), n


In [ ]:
stock_files = sorted(
    p for p in DATA_DIR.rglob("*.parquet")
    if "results" not in p.parts and "indices" not in p.parts
)
print("Stock parquet files:", len(stock_files))
if stock_files:
    sample = pd.read_parquet(stock_files[0])
    print("Sample file:", stock_files[0].name, "rows:", len(sample), "cols:", list(sample.columns)[:12])
    print("Unique dates in sample file:", pd.to_datetime(sample.iloc[:, 0], errors="coerce").nunique() if len(sample) else 0)
    del sample

panel_src = None
if FEATURES_PATH.exists():
    panel_src = "features"
    print("Using existing feature panel:", FEATURES_PATH)
elif not stock_files:
    raise FileNotFoundError(f"No stock parquet under {DATA_DIR}")
else:
    if PANEL_PATH.exists():
        PANEL_PATH.unlink()
    writer = None
    n_ok = n_err = 0
    peek_dates = []
    for i, f in enumerate(stock_files, 1):
        try:
            raw = pd.read_parquet(f)
            x = normalize_ohlc(raw, f.name)
            del raw
            if x.empty:
                n_err += 1
                continue
            if len(peek_dates) < 5:
                peek_dates.append((f.name, int(x["Date"].nunique()), len(x), int(x["Symbol"].nunique())))
            table = pa.Table.from_pandas(x[["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"]], schema=PANEL_SCHEMA, preserve_index=False)
            del x
            if writer is None:
                writer = pq.ParquetWriter(PANEL_PATH, PANEL_SCHEMA, compression="zstd")
            writer.write_table(table)
            del table
            n_ok += 1
            if i % 250 == 0:
                gc.collect()
                print(f"stitched {i}/{len(stock_files)} files")
        except Exception as e:
            n_err += 1
            if n_err <= 8:
                print("skip", f.name, e)
    if writer is not None:
        writer.close()
    gc.collect()
    print("Stitched files OK:", n_ok, "errors:", n_err)
    print("Per-file layout (name, unique_dates, rows, symbols):", peek_dates)
    panel_src = "stitched"

def read_panel_cols(path):
    have = set(pq.ParquetFile(path).schema.names)
    cols = [c for c in ["Date", "Symbol", "Open", "High", "Low", "Close", "Volume"] if c in have]
    return pd.read_parquet(path, columns=cols)

if panel_src == "features":
    panel = read_panel_cols(FEATURES_PATH)
else:
    panel = read_panel_cols(PANEL_PATH)
for c in ["Open", "Volume"]:
    if c not in panel.columns:
        panel[c] = np.nan

panel["Date"] = pd.to_datetime(panel["Date"])
panel["Symbol"] = panel["Symbol"].astype("string")
print("Panel rows:", len(panel), "symbols:", panel["Symbol"].nunique(),
      "dates:", panel["Date"].min(), "→", panel["Date"].max())
print("Median bars per symbol:", float(panel.groupby("Symbol", observed=True).size().median()))


In [ ]:
rows = []
mult_rows = []
for sym, g in panel.groupby("Symbol", sort=False, observed=True):
    g = g.sort_values("Date").drop_duplicates("Date", keep="last")
    rec, max_mult, n = first_target_run(
        g["Date"].to_numpy(),
        g["Close"].to_numpy(),
        g["High"].to_numpy(),
        g["Low"].to_numpy(),
        str(sym),
    )
    mult_rows.append({"Symbol": str(sym), "Bars": n, "MaxMultipleFromExpandingMin": max_mult})
    if rec is not None:
        rows.append(rec)

gc.collect()

mults = pd.DataFrame(mult_rows)
hits = pd.DataFrame(rows)
print("Symbols scanned:", len(mults))
print("Max multiple percentiles:")
print(mults["MaxMultipleFromExpandingMin"].replace(0, np.nan).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
for m, label in [(2, "100%"), (11, "1,000%"), (51, "5,000%"), (101, "10,000%")]:
    n = int((mults["MaxMultipleFromExpandingMin"] >= m).sum())
    print(f"  names with ≥ {label} from expanding-min close ({m:.0f}x): {n}")
print("Hits at +10,000%:", len(hits))
mults.sort_values("MaxMultipleFromExpandingMin", ascending=False).head(30).to_csv(OUT_DIR / "top_multiples_all_names.csv", index=False)


## Results

Only names that actually printed a +10,000% close from the expanding-min entry. Sorted by calendar days.


In [ ]:
if hits.empty:
    print("Still no +10,000% names. Inspect top multiples CSV — if 10x/50x counts are large but 101x is 0, the data simply never reached 101x from a prior low.")
    top = mults.sort_values("MaxMultipleFromExpandingMin", ascending=False).head(40)
    display(top)
else:
    hits = hits.sort_values(["CalendarDays", "MaxDrawdownLow"]).reset_index(drop=True)
    hits.to_csv(OUT_DIR / "stocks_10000pct_from_expanding_min.csv", index=False)
    hits.to_parquet(OUT_DIR / "stocks_10000pct_from_expanding_min.parquet", index=False)
    view = hits[[
        "Symbol", "EntryDate", "HitDate", "EntryClose", "HitClose",
        "ReturnPct", "Multiple", "TradingDays", "CalendarDays", "Years",
        "MaxDrawdownClose", "MaxDrawdownLow", "WorstDDDate", "LowEntryFlag",
        "MaxMultipleFromExpandingMin",
    ]].copy()
    show = view.copy()
    show["MaxDrawdownClose"] = show["MaxDrawdownClose"].map(lambda v: f"{v:.2%}")
    show["MaxDrawdownLow"] = show["MaxDrawdownLow"].map(lambda v: f"{v:.2%}")
    show["ReturnPct"] = show["ReturnPct"].map(lambda v: f"{v:,.0f}%")
    display(show)
    print("Count:", len(hits))
    print("Median calendar days:", float(hits["CalendarDays"].median()))
    print("Median trading days:", float(hits["TradingDays"].median()))
    print("Median max DD (close):", f"{hits['MaxDrawdownClose'].median():.2%}")
    print("Median max DD (low):", f"{hits['MaxDrawdownLow'].median():.2%}")
    print("Worst max DD (low):", f"{hits['MaxDrawdownLow'].min():.2%}",
          hits.loc[hits["MaxDrawdownLow"].idxmin(), "Symbol"])
    print("Fastest calendar days:", int(hits["CalendarDays"].min()),
          hits.loc[hits["CalendarDays"].idxmin(), "Symbol"])


In [ ]:
if not hits.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist(hits["CalendarDays"], bins=min(40, max(5, len(hits))), color="steelblue", edgecolor="white")
    axes[0].set_title("Calendar days to +10,000%")
    axes[1].hist(hits["MaxDrawdownLow"] * 100, bins=min(40, max(5, len(hits))), color="indianred", edgecolor="white")
    axes[1].set_title("Max DD on the path (lows)")
    axes[2].scatter(hits["CalendarDays"], hits["MaxDrawdownLow"] * 100, alpha=0.7)
    axes[2].set_xlabel("Calendar days")
    axes[2].set_ylabel("Max DD %")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "days_and_drawdown.png", dpi=120)
    plt.show()
    plt.close(fig)
    print("CSV:", OUT_DIR / "stocks_10000pct_from_expanding_min.csv")


## How to read a row

Bought `EntryClose` on `EntryDate` (lowest close up to then). First +10,000% close is `HitDate`. `MaxDrawdownLow` / `MaxDrawdownClose` is the worst peak-to-trough **on that path**.


## Pre-rally pattern (history available at the trough only)

`EntryDate` is a **new expanding-min close** by construction, so “it was at a low” is not a finding. The question is whether hundred-baggers **got there the same way**, and whether that path differs from **new lows that never became 101×**.

All features use bars `≤ EntryIndex`. Nothing after the trough is used to describe the setup.


In [ ]:
def rolling_mean(a, n):
    a = np.asarray(a, dtype=np.float64)
    if len(a) < n or n <= 0:
        return np.nan
    return float(np.nanmean(a[-n:]))

def rolling_std(a, n):
    a = np.asarray(a, dtype=np.float64)
    if len(a) < n:
        return np.nan
    x = a[-n:]
    x = x[np.isfinite(x)]
    if len(x) < max(5, n // 4):
        return np.nan
    return float(np.std(x, ddof=1))

def sma(a, n):
    a = np.asarray(a, dtype=np.float64)
    if len(a) < n:
        return np.nan
    return float(np.nanmean(a[-n:]))

def pre_trough_features(dates, o, h, l, c, v, i):
    """Features at index i using only [:i+1]."""
    dates = pd.to_datetime(np.asarray(dates))
    o = np.asarray(o, dtype=np.float64)
    h = np.asarray(h, dtype=np.float64)
    l = np.asarray(l, dtype=np.float64)
    c = np.asarray(c, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    n = i + 1
    if n < 5:
        return {}
    c0, h0, l0, v0 = c[:n], h[:n], l[:n], v[:n]
    px = c0[-1]
    rng = np.where(np.isfinite(h0) & np.isfinite(l0), h0 - l0, np.nan)
    # Prior peak strictly before the trough bar
    if n >= 2:
        peak_i = int(np.nanargmax(c0[:-1]))
        prior_peak = float(c0[peak_i])
        days_since_peak = int((dates[i] - dates[peak_i]).days)
        bars_since_peak = int(i - peak_i)
        dd_from_peak = px / prior_peak - 1.0 if prior_peak > 0 else np.nan
    else:
        peak_i = 0
        prior_peak = px
        days_since_peak = 0
        bars_since_peak = 0
        dd_from_peak = 0.0
    vol20 = rolling_std(np.diff(np.log(np.clip(c0, 1e-12, None))), 20)
    vol60 = rolling_std(np.diff(np.log(np.clip(c0, 1e-12, None))), 60)
    vol120 = rolling_std(np.diff(np.log(np.clip(c0, 1e-12, None))), 120)
    atrp20 = rolling_mean(rng / np.clip(c0, 1e-12, None), 20)
    atrp120 = rolling_mean(rng / np.clip(c0, 1e-12, None), 120)
    v20 = rolling_mean(v0, 20)
    v120 = rolling_mean(v0, 120)
    v_climax = (np.nanmax(v0[-20:]) / v120) if n >= 20 and np.isfinite(v120) and v120 > 0 else np.nan
    sma20 = sma(c0, 20)
    sma50 = sma(c0, 50)
    sma200 = sma(c0, 200)
    sma200_prev = sma(c0[:-20], 200) if n > 220 else np.nan
    # Waterfall: share of last 120 days that made a 20-day low
    waterfall = np.nan
    if n >= 40:
        last = c0[-120:] if n >= 120 else c0
        hits_ll = 0
        for k in range(20, len(last)):
            if last[k] <= np.nanmin(last[k-20:k]) + 1e-12:
                hits_ll += 1
        waterfall = hits_ll / max(len(last) - 20, 1)
    # Base tightness last 60 vs 252 range
    rng60 = (np.nanmax(h0[-60:]) / np.nanmin(l0[-60:]) - 1.0) if n >= 60 else np.nan
    rng252 = (np.nanmax(h0[-252:]) / np.nanmin(l0[-252:]) - 1.0) if n >= 252 else np.nan
    ret20 = px / c0[-21] - 1.0 if n > 21 else np.nan
    ret60 = px / c0[-61] - 1.0 if n > 61 else np.nan
    ret252 = px / c0[-253] - 1.0 if n > 253 else np.nan
    return {
        "TroughDate": pd.Timestamp(dates[i]),
        "TroughClose": float(px),
        "BarsBefore": int(i),
        "PriorPeakClose": prior_peak,
        "DaysSincePriorPeak": days_since_peak,
        "BarsSincePriorPeak": bars_since_peak,
        "DrawdownFromPriorPeak": dd_from_peak,
        "Vol20": vol20,
        "Vol60": vol60,
        "Vol120": vol120,
        "VolCompression": (vol20 / vol120) if vol120 and np.isfinite(vol20) and vol120 > 0 else np.nan,
        "ATRp20": atrp20,
        "ATRp120": atrp120,
        "RangeCompression": (atrp20 / atrp120) if atrp120 and atrp120 > 0 else np.nan,
        "Volume20": v20,
        "Volume120": v120,
        "VolumeDryUp": (v20 / v120) if v120 and v120 > 0 else np.nan,
        "VolumeClimax20": v_climax,
        "CloseVsSMA20": px / sma20 - 1.0 if sma20 else np.nan,
        "CloseVsSMA50": px / sma50 - 1.0 if sma50 else np.nan,
        "CloseVsSMA200": px / sma200 - 1.0 if sma200 else np.nan,
        "SMA200Slope20": (sma200 / sma200_prev - 1.0) if sma200_prev else np.nan,
        "Range60": rng60,
        "Range252": rng252,
        "Return20": ret20,
        "Return60": ret60,
        "Return252": ret252,
        "WaterfallLowRate120": waterfall,
        "TroughMonth": int(pd.Timestamp(dates[i]).month),
        "TroughYear": int(pd.Timestamp(dates[i]).year),
        "TroughWeekday": int(pd.Timestamp(dates[i]).dayofweek),
        "ListingToTroughDays": int((dates[i] - dates[0]).days),
    }

def load_nifty_close():
    if not NIFTY50_PATH.exists():
        return None
    n = pd.read_parquet(NIFTY50_PATH)
    cols = {str(c).lower().strip(): c for c in n.columns}
    dcol = next((cols[k] for k in ["date", "datetime", "timestamp"] if k in cols), None)
    ccol = next((cols[k] for k in ["close", "adj close", "adj_close"] if k in cols), None)
    if dcol is None or ccol is None:
        return None
    out = pd.DataFrame({
        "Date": pd.to_datetime(n[dcol], errors="coerce"),
        "NiftyClose": pd.to_numeric(n[ccol], errors="coerce"),
    }).dropna()
    if getattr(out["Date"].dt, "tz", None) is not None:
        out["Date"] = out["Date"].dt.tz_convert("UTC").dt.tz_localize(None)
    out = out.sort_values("Date").drop_duplicates("Date")
    out["NiftySMA200"] = out["NiftyClose"].rolling(200, min_periods=200).mean()
    out["NiftyRet60"] = out["NiftyClose"] / out["NiftyClose"].shift(60) - 1.0
    out["NiftyRet252"] = out["NiftyClose"] / out["NiftyClose"].shift(252) - 1.0
    return out.set_index("Date")

nifty = load_nifty_close()
print("Nifty rows:", 0 if nifty is None else len(nifty))


In [ ]:
# Rebuild per-symbol arrays once for pattern + controls (panel still in memory).
if "panel" not in globals() or panel is None or len(panel) == 0:
    raise RuntimeError("Panel was dropped. Re-run the stitch cell.")

hit_syms = set(hits["Symbol"].astype(str)) if hits is not None and len(hits) else set()
feat_hits = []
feat_ctrl = []
rng_ctrl = np.random.default_rng(42)

for sym, g in panel.groupby("Symbol", sort=False, observed=True):
    g = g.sort_values("Date").drop_duplicates("Date", keep="last")
    dates = g["Date"].to_numpy()
    o = g["Open"].to_numpy() if "Open" in g.columns else np.full(len(g), np.nan)
    h = g["High"].to_numpy()
    l = g["Low"].to_numpy()
    c = g["Close"].to_numpy(dtype=np.float64)
    v = g["Volume"].to_numpy() if "Volume" in g.columns else np.full(len(g), np.nan)
    n = len(c)
    if n < MIN_BARS:
        continue
    rec, max_mult, _ = first_target_run(dates, c, h, l, str(sym))
    if rec is not None:
        i = int(rec["EntryIndex"])
        f = pre_trough_features(dates, o, h, l, c, v, i)
        f["Symbol"] = str(sym)
        f["Group"] = "hit_101x"
        f["LaterMaxMultiple"] = float(max_mult)
        feat_hits.append(f)
        continue
    # Controls: up to two new expanding-mins that never reached 101x
    rmin = np.inf
    low_idx = []
    for i in range(n):
        px = c[i]
        if not np.isfinite(px) or px <= 0:
            continue
        if px < rmin:
            rmin = px
            if i >= 60:
                low_idx.append(i)
    if not low_idx:
        continue
    pick = []
    pick.append(low_idx[-1])
    if len(low_idx) >= 3:
        pick.append(low_idx[len(low_idx) // 2])
    pick = list(dict.fromkeys(pick))
    for i in pick:
        f = pre_trough_features(dates, o, h, l, c, v, i)
        f["Symbol"] = str(sym)
        f["Group"] = "failed_new_low"
        f["LaterMaxMultiple"] = float(max_mult)
        feat_ctrl.append(f)

pre = pd.DataFrame(feat_hits)
ctrl = pd.DataFrame(feat_ctrl)
print("Hit troughs with features:", len(pre), "control new-lows:", len(ctrl))

def _asof_ts(s):
    s = pd.to_datetime(s, errors="coerce")
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_convert("UTC").dt.tz_localize(None)
    return s

def attach_nifty(df):
    if nifty is None or df.empty or "TroughDate" not in df.columns:
        return df
    x = df.copy()
    x["TroughDate"] = _asof_ts(x["TroughDate"])
    x = x.dropna(subset=["TroughDate"])
    n = nifty.reset_index().rename(columns={"Date": "TroughDate"})
    n["TroughDate"] = _asof_ts(n["TroughDate"])
    n = n.dropna(subset=["TroughDate"]).sort_values("TroughDate")
    if x.empty or n.empty:
        return x
    x = pd.merge_asof(
        x.sort_values("TroughDate"),
        n,
        on="TroughDate",
        direction="backward",
    )
    x["NiftyAbove200"] = x["NiftyClose"] > x["NiftySMA200"]
    return x

pre = attach_nifty(pre)
ctrl = attach_nifty(ctrl)
pre.to_csv(OUT_DIR / "pre_rally_features_hits.csv", index=False)
ctrl.to_csv(OUT_DIR / "pre_rally_features_controls.csv", index=False)


In [ ]:
NUM_COLS = [
    "DaysSincePriorPeak", "BarsSincePriorPeak", "DrawdownFromPriorPeak",
    "Vol20", "Vol60", "VolCompression", "ATRp20", "RangeCompression",
    "VolumeDryUp", "VolumeClimax20",
    "CloseVsSMA20", "CloseVsSMA50", "CloseVsSMA200", "SMA200Slope20",
    "Range60", "Range252", "Return20", "Return60", "Return252",
    "WaterfallLowRate120", "ListingToTroughDays", "TroughClose",
    "NiftyRet60", "NiftyRet252",
]
NUM_COLS = [c for c in NUM_COLS if c in pre.columns or c in ctrl.columns]

def summarize_group(df, name):
    rows = []
    for c in NUM_COLS:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        rows.append({
            "Feature": c,
            "Group": name,
            "N": int(s.notna().sum()),
            "Median": float(s.median()),
            "P25": float(s.quantile(0.25)),
            "P75": float(s.quantile(0.75)),
            "Mean": float(s.mean()),
        })
    return pd.DataFrame(rows)

cmp = pd.concat([summarize_group(pre, "hit_101x"), summarize_group(ctrl, "failed_new_low")], ignore_index=True)
wide = cmp.pivot(index="Feature", columns="Group", values=["Median", "P25", "P75", "N"])
# flatten
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()
if "Median_hit_101x" in wide.columns and "Median_failed_new_low" in wide.columns:
    wide["HitMinusControlMedian"] = wide["Median_hit_101x"] - wide["Median_failed_new_low"]
display(wide.sort_values("Feature"))
wide.to_csv(OUT_DIR / "pre_rally_hit_vs_control.csv", index=False)

print("\nShare of hit troughs vs failed new-lows")
rules = [
    ("Drawdown from prior peak ≤ −70%", "DrawdownFromPriorPeak", lambda s: s <= -0.70),
    ("Drawdown from prior peak ≤ −90%", "DrawdownFromPriorPeak", lambda s: s <= -0.90),
    ("≥ 2 years since prior peak", "DaysSincePriorPeak", lambda s: s >= 365 * 2),
    ("≥ 5 years since prior peak", "DaysSincePriorPeak", lambda s: s >= 365 * 5),
    ("Below SMA200", "CloseVsSMA200", lambda s: s < 0),
    ("SMA200 still falling (20d)", "SMA200Slope20", lambda s: s < 0),
    ("Vol compressed (20/120 < 0.7)", "VolCompression", lambda s: s < 0.70),
    ("Range compressed (ATR20/ATR120 < 0.7)", "RangeCompression", lambda s: s < 0.70),
    ("Volume dry-up (20/120 < 0.7)", "VolumeDryUp", lambda s: s < 0.70),
    ("Volume climax (≥ 3× 120d avg in last 20d)", "VolumeClimax20", lambda s: s >= 3.0),
    ("12m return ≤ −50%", "Return252", lambda s: s <= -0.50),
    ("Waterfall (new 20d lows often)", "WaterfallLowRate120", lambda s: s >= 0.15),
    ("Penny-ish trough close < ₹5", "TroughClose", lambda s: s < 5),
    ("Trough close < ₹20", "TroughClose", lambda s: s < 20),
]
if nifty is not None and "NiftyAbove200" in pre.columns:
    rules.append(("Nifty below SMA200", "NiftyAbove200", lambda s: s == False))
    rules.append(("Nifty 12m return < 0", "NiftyRet252", lambda s: s < 0))

rule_rows = []
for label, col, fn in rules:
    row = {"Rule": label}
    for name, df in [("hit_101x", pre), ("failed_new_low", ctrl)]:
        if col not in df.columns or df.empty:
            row[name] = np.nan
            continue
        s = pd.to_numeric(df[col], errors="coerce") if df[col].dtype != bool else df[col]
        s = s.dropna()
        row[name] = float(fn(s).mean()) if len(s) else np.nan
    rule_rows.append(row)
rule_df = pd.DataFrame(rule_rows)
display(rule_df)
rule_df.to_csv(OUT_DIR / "pre_rally_rule_rates.csv", index=False)


In [ ]:
if pre.empty:
    print("No hit features. Re-run the scan until hits is non-empty.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    pairs = [
        ("DrawdownFromPriorPeak", "Drawdown from prior peak"),
        ("DaysSincePriorPeak", "Days since prior peak"),
        ("VolCompression", "Vol20 / Vol120"),
        ("VolumeDryUp", "Volume20 / Volume120"),
        ("CloseVsSMA200", "Close vs SMA200"),
        ("Return252", "Trailing 12m return"),
    ]
    for ax, (col, title) in zip(axes.ravel(), pairs):
        if col in pre.columns:
            ax.hist(pd.to_numeric(pre[col], errors="coerce").dropna(), bins=20, alpha=0.7, label="101x hits", color="steelblue")
        if col in ctrl.columns and len(ctrl):
            ax.hist(pd.to_numeric(ctrl[col], errors="coerce").dropna(), bins=20, alpha=0.35, label="failed new lows", color="gray")
        ax.set_title(title)
        ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "pre_rally_distributions.png", dpi=120)
    plt.show()
    plt.close(fig)

    print("Trough year counts (hits):")
    display(pre["TroughYear"].value_counts().sort_index())
    print("Trough month counts (hits):")
    display(pre["TroughMonth"].value_counts().sort_index())

    # Shared-pattern writeup from medians — descriptive, not a trading rule.
    def med(df, col):
        return float(pd.to_numeric(df[col], errors="coerce").median()) if col in df.columns and len(df) else np.nan

    lines = []
    lines.append(f"N hundred-bagger troughs: {len(pre)}; N failed new-lows: {len(ctrl)}")
    lines.append(
        f"Hits: median decline from prior peak {med(pre,'DrawdownFromPriorPeak'):.1%} "
        f"over {med(pre,'DaysSincePriorPeak'):.0f} calendar days "
        f"({med(pre,'BarsSincePriorPeak'):.0f} sessions)."
    )
    if len(ctrl):
        lines.append(
            f"Failed new-lows: median decline {med(ctrl,'DrawdownFromPriorPeak'):.1%} "
            f"over {med(ctrl,'DaysSincePriorPeak'):.0f} days."
        )
        def lift(col, hit_dir="lower"):
            h, c = med(pre, col), med(ctrl, col)
            return h, c
        h, c = lift("DrawdownFromPriorPeak")
        if pd.notna(h) and pd.notna(c) and h < c:
            lines.append("Hits typically fell farther from the prior peak than ordinary new lows.")
        h, c = lift("DaysSincePriorPeak")
        if pd.notna(h) and pd.notna(c) and h > c:
            lines.append("Hits typically spent longer declining / washing out before the trough.")
        h, c = lift("VolumeDryUp")
        if pd.notna(h) and pd.notna(c) and h < c:
            lines.append("Hits more often showed volume dry-up into the low (20d volume vs 120d).")
        elif pd.notna(h) and pd.notna(c) and h > c:
            lines.append("Hits more often showed heavier volume into the low (possible capitulation), not dry-up.")
        h, c = lift("VolCompression")
        if pd.notna(h) and pd.notna(c) and h < c:
            lines.append("Hits more often compressed volatility into the trough.")
        elif pd.notna(h) and pd.notna(c) and h > c:
            lines.append("Hits more often still had elevated volatility at the trough (crisis/waterfall), not a tight base.")
        h, c = lift("TroughClose")
        if pd.notna(h) and pd.notna(c) and h < c:
            lines.append("Hits printed the 101× from a lower rupee price than typical failed new-lows (small-price / deep-distress names).")
    if "NiftyAbove200" in pre.columns and pre["NiftyAbove200"].notna().any():
        share = 1.0 - float(pre["NiftyAbove200"].mean())
        lines.append(f"Share of hit troughs with Nifty below SMA200: {share:.1%}.")
    lines.append("Do not treat this as an entry model. It is a description of how 101× names looked at a sample low vs other sample lows.")
    print("\n".join(lines))
    (OUT_DIR / "pre_rally_pattern_summary.txt").write_text("\n".join(lines) + "\n")

del panel
gc.collect()
print("Wrote", OUT_DIR)


---
# V2 — Matched-control and post-trough research

**Design:** one 101× trough per hit symbol; at most one eligible expanding-min trough per non-hit symbol. Require 252 subsequent observed bars for both groups, and match controls on trough year, price, prior drawdown and listing age (plus prior-peak duration). This is a **retrospective case-control comparison**, not an estimate of the probability of a 101× outcome: hit labels use future prices. Controls are non-hit **as of the dataset end**, not proven permanent failures. Survivorship, ticker reuse, split/bonus adjustment and missing/delisted securities can materially bias results.

Forward returns and features are computed strictly at each specified landmark. A 252-day feature must never be treated as known at day 20. The original first-target-run uses an expanding-min entry, not an executable next-open trade.

In [ ]:
# V2 settings: retain original DATA_DIR and panel from preceding cells.
V2_DIR = OUT_DIR / "v2"
V2_DIR.mkdir(parents=True, exist_ok=True)
LANDMARKS = (20, 60, 120, 252)
MIN_FOLLOWUP = max(LANDMARKS)
MATCH_K = 3
MATCH_YEAR_TOL = 1
MATCH_PRICE_LOG_TOL = np.log(4.0)  # within 4x trough price
MATCH_DD_TOL = 0.25                 # within 25 percentage points of prior drawdown
RANDOM_SEED = 42
print("V2 outputs:", V2_DIR)
print("Dataset end:", panel["Date"].max(), "| required forward bars:", MIN_FOLLOWUP)


## 1. Build comparable cohorts

Hit observations are the original hit troughs **only if they have a complete 252-bar follow-up**. For each non-hit symbol, select one eligible expanding-min trough (deterministically, nearest to the median hit trough year). This avoids the original two-controls-per-symbol imbalance. A failed-new-low control can still have a substantial rebound, but never achieved 101× under the original full-history definition.

Matching uses only pre-trough attributes. Matching without replacement and exact/near-exact calendar year reduces obvious era imbalance. Unmatched hits remain visible in the audit export and are excluded from the paired comparison.

In [ ]:
def v2_arrays(g):
    g = g.sort_values("Date").drop_duplicates("Date", keep="last")
    return (g["Date"].to_numpy(),
            g["Open"].to_numpy(dtype=float),
            g["High"].to_numpy(dtype=float),
            g["Low"].to_numpy(dtype=float),
            g["Close"].to_numpy(dtype=float),
            g["Volume"].to_numpy(dtype=float))

def v2_expanding_lows(c, min_index=MIN_BARS, followup=MIN_FOLLOWUP):
    running_min = np.inf
    eligible = []
    for i, px in enumerate(c):
        if not np.isfinite(px) or px <= 0:
            continue
        if px < running_min:
            running_min = px
            if i >= min_index and i + followup < len(c):
                eligible.append(i)
    return eligible

hit_lookup = {str(r.Symbol): r for r in hits.itertuples(index=False)} if not hits.empty else {}
hit_years = pd.to_datetime(hits["EntryDate"]).dt.year if not hits.empty else pd.Series(dtype=int)
target_year = float(hit_years.median()) if len(hit_years) else 2020
v2_hit_rows, v2_control_rows, v2_exclusions = [], [], []
for sym, g in panel.groupby("Symbol", sort=False, observed=True):
    dates, o, h, l, c, v = v2_arrays(g)
    sym = str(sym)
    if len(c) < MIN_BARS + MIN_FOLLOWUP:
        continue
    if sym in hit_lookup:
        i = int(hit_lookup[sym].EntryIndex)
        if i < MIN_BARS or i + MIN_FOLLOWUP >= len(c):
            v2_exclusions.append({"Symbol": sym, "Reason": "hit lacks 252-bar follow-up or prehistory"})
            continue
        f = pre_trough_features(dates, o, h, l, c, v, i)
        f.update(Symbol=sym, Group="hit_101x", TroughIndex=i)
        v2_hit_rows.append(f)
    else:
        eligible = v2_expanding_lows(c)
        if not eligible:
            continue
        # Pick one control per symbol; choice uses its observed lows and calendar era.
        i = min(eligible, key=lambda j: (abs(pd.Timestamp(dates[j]).year - target_year), -j))
        f = pre_trough_features(dates, o, h, l, c, v, i)
        f.update(Symbol=sym, Group="failed_new_low", TroughIndex=i)
        v2_control_rows.append(f)

v2_hits = attach_nifty(pd.DataFrame(v2_hit_rows))
v2_controls = attach_nifty(pd.DataFrame(v2_control_rows))
pd.DataFrame(v2_exclusions, columns=["Symbol", "Reason"]).to_csv(V2_DIR / "excluded_hits.csv", index=False)
print("Follow-up eligible hits:", len(v2_hits), "| unique controls:", len(v2_controls))
if v2_hits.empty or v2_controls.empty:
    raise RuntimeError("Insufficient hit/control observations with 252-bar follow-up")


In [ ]:
# Caliper-based nearest-neighbor matching, no replacement.
# Only pre-trough features enter the matching distance.
match_cols = ["DrawdownFromPriorPeak", "DaysSincePriorPeak", "ListingToTroughDays"]
match_scale = {k: max(float(pd.to_numeric(v2_controls[k], errors="coerce").std()), 1.0)
               for k in match_cols}
match_scale["DrawdownFromPriorPeak"] = max(match_scale["DrawdownFromPriorPeak"], 0.1)
used = set()
matched = []
audit = []
for _, hit in v2_hits.sort_values(["TroughDate", "Symbol"]).iterrows():
    pool = v2_controls.loc[~v2_controls["Symbol"].isin(used)].copy()
    pool = pool.loc[(pool["TroughYear"] - hit["TroughYear"]).abs() <= MATCH_YEAR_TOL]
    pool = pool.loc[(np.log(pool["TroughClose"] / hit["TroughClose"])).abs() <= MATCH_PRICE_LOG_TOL]
    pool = pool.loc[(pool["DrawdownFromPriorPeak"] - hit["DrawdownFromPriorPeak"]).abs() <= MATCH_DD_TOL]
    pool = pool.dropna(subset=match_cols + ["TroughClose"])
    if pool.empty:
        audit.append({"HitSymbol": hit["Symbol"], "HitDate": hit["TroughDate"], "MatchedControls": 0})
        continue
    distance = ((pool["TroughYear"] - hit["TroughYear"]) / 1.0) ** 2
    distance += (np.log(pool["TroughClose"] / hit["TroughClose"]) / 1.0) ** 2
    for k in match_cols:
        distance += ((pool[k] - hit[k]) / match_scale[k]) ** 2
    chosen = pool.assign(MatchDistance=distance).nsmallest(MATCH_K, "MatchDistance")
    audit.append({"HitSymbol": hit["Symbol"], "HitDate": hit["TroughDate"], "MatchedControls": len(chosen)})
    for _, ctrl_row in chosen.iterrows():
        used.add(ctrl_row["Symbol"])
        matched.append({"MatchID": hit["Symbol"] + "|" + str(pd.Timestamp(hit["TroughDate"]).date()),
                        "HitSymbol": hit["Symbol"], "ControlSymbol": ctrl_row["Symbol"],
                        "HitDate": hit["TroughDate"], "ControlDate": ctrl_row["TroughDate"],
                        "MatchDistance": ctrl_row["MatchDistance"]})
matches = pd.DataFrame(matched, columns=["MatchID", "HitSymbol", "ControlSymbol", "HitDate", "ControlDate", "MatchDistance"])
match_audit = pd.DataFrame(audit)
matches.to_csv(V2_DIR / "matched_pairs.csv", index=False)
match_audit.to_csv(V2_DIR / "match_audit.csv", index=False)
print("Matched hit symbols:", matches["HitSymbol"].nunique(), "/", len(v2_hits),
      "| matched control symbols:", matches["ControlSymbol"].nunique())
if matches.empty:
    raise RuntimeError("No caliper matches. Review V2 matching tolerances and cohort eligibility.")
matched_hit_symbols = set(matches["HitSymbol"])
matched_control_symbols = set(matches["ControlSymbol"])
cohort = pd.concat([v2_hits[v2_hits.Symbol.isin(matched_hit_symbols)],
                    v2_controls[v2_controls.Symbol.isin(matched_control_symbols)]], ignore_index=True)
cohort.to_parquet(V2_DIR / "matched_cohort_pre_trough.parquet", index=False)


## 2. Forward landmark features (no look-ahead within each landmark)

At day *d*, compute only trough-to-day-*d* price and volume history. SMA recovery uses the full historical series **up to day d**; breakout is measured against the previous 20-day high and the original trough-day 60-day high. Missing open/volume/high/low remains missing rather than being silently filled. The 252-bar follow-up restriction is a research-cohort selection, not a live eligibility rule.

In [ ]:
def v2_safe_ratio(a, b):
    return float(a / b) if np.isfinite(a) and np.isfinite(b) and b > 0 else np.nan

def v2_landmark(dates, o, h, l, c, v, i, day):
    j = i + day
    if j >= len(c) or i < 60:
        return None
    base, now = float(c[i]), float(c[j])
    hist = c[:j + 1]
    seg = c[i:j + 1]
    vv = v[i + 1:j + 1]
    rr = np.diff(np.log(np.clip(seg, 1e-12, None)))
    ref_volume = np.nanmean(v[max(0, i-119):i+1]) if i >= 19 else np.nan
    ref_vol = np.std(np.diff(np.log(c[i-120:i+1])), ddof=1) if i >= 120 else np.nan
    day_vol = np.std(rr[-20:], ddof=1) if len(rr) >= 20 else np.nan
    sma20 = float(np.mean(hist[-20:])) if len(hist) >= 20 else np.nan
    sma50 = float(np.mean(hist[-50:])) if len(hist) >= 50 else np.nan
    sma200 = float(np.mean(hist[-200:])) if len(hist) >= 200 else np.nan
    past20_high = np.nanmax(h[max(i, j-20):j]) if j > i else np.nan
    trough_ref_high = np.nanmax(h[i-59:i+1])
    prior_low = np.minimum.accumulate(seg[:-1]) if len(seg) > 1 else np.array([])
    higher_lows = int(np.sum(seg[1:] > prior_low)) if len(prior_low) else 0
    return {
        "Day": day, "AsOfDate": pd.Timestamp(dates[j]), "AsOfIndex": j,
        "Return": now / base - 1, "Close": now,
        "MaxReturnToDate": float(np.nanmax(seg / base - 1)),
        "MinReturnToDate": float(np.nanmin(seg / base - 1)),
        "MaxDrawdownToDate": float(np.nanmin(seg / np.maximum.accumulate(seg) - 1)),
        "AboveSMA20": float(now > sma20) if np.isfinite(sma20) else np.nan,
        "AboveSMA50": float(now > sma50) if np.isfinite(sma50) else np.nan,
        "AboveSMA200": float(now > sma200) if np.isfinite(sma200) else np.nan,
        "SMA20Slope20": v2_safe_ratio(sma20, float(np.mean(hist[-40:-20])) if len(hist) >= 40 else np.nan) - 1,
        "VolumeVsPre120": v2_safe_ratio(float(np.nanmean(vv[-min(20, len(vv)):])), ref_volume),
        "Vol20VsPre120": v2_safe_ratio(day_vol, ref_vol),
        "HigherThanPrevious20High": float(now > past20_high) if np.isfinite(past20_high) else np.nan,
        "AboveTrough60High": float(now > trough_ref_high) if np.isfinite(trough_ref_high) else np.nan,
        "BarsAboveRunningPostLow": higher_lows,
    }

cohort_keys = set(zip(cohort.Symbol.astype(str), pd.to_datetime(cohort.TroughDate)))
cohort_lookup = {(str(r.Symbol), pd.Timestamp(r.TroughDate)): r for r in cohort.itertuples(index=False)}
forward_rows = []
for sym, g in panel.groupby("Symbol", sort=False, observed=True):
    sym = str(sym)
    if sym not in matched_hit_symbols and sym not in matched_control_symbols:
        continue
    dates, o, h, l, c, v = v2_arrays(g)
    key_candidates = [key for key in cohort_keys if key[0] == sym]
    for key in key_candidates:
        rec = cohort_lookup[key]
        i = int(rec.TroughIndex)
        for day in LANDMARKS:
            f = v2_landmark(dates, o, h, l, c, v, i, day)
            if f is not None:
                f.update(Symbol=sym, Group=rec.Group, TroughDate=key[1], TroughClose=float(c[i]))
                forward_rows.append(f)
forward = pd.DataFrame(forward_rows)
forward.to_parquet(V2_DIR / "post_trough_landmarks.parquet", index=False)
forward.to_csv(V2_DIR / "post_trough_landmarks.csv", index=False)
print("Landmark observations:", len(forward), "| by day:", forward.groupby("Day").size().to_dict())


## 3. Matched comparison and uncertainty

Use **equal weight per hit**: average each hit's matched controls first, then compare hit versus its matched-control mean. Report descriptive differences and bootstrap confidence intervals over hit-level pairs (not individual control rows). These are exploratory intervals, not causal effects or live predictive performance.

In [ ]:
METRICS = ["Return", "MaxReturnToDate", "MinReturnToDate", "MaxDrawdownToDate",
           "AboveSMA20", "AboveSMA50", "AboveSMA200", "SMA20Slope20",
           "VolumeVsPre120", "Vol20VsPre120", "HigherThanPrevious20High", "AboveTrough60High"]
metric_rows, pair_rows = [], []
rng = np.random.default_rng(RANDOM_SEED)
for day in LANDMARKS:
    fd = forward.loc[forward.Day == day].set_index("Symbol")
    for metric in METRICS:
        diffs, hv, cv = [], [], []
        for hit_sym, pair_group in matches.groupby("HitSymbol", sort=False):
            if hit_sym not in fd.index or metric not in fd:
                continue
            a = pd.to_numeric(pd.Series([fd.loc[hit_sym, metric]]), errors="coerce").iloc[0]
            controls = [s for s in pair_group.ControlSymbol if s in fd.index]
            bvals = pd.to_numeric(fd.loc[controls, metric], errors="coerce").dropna() if controls else pd.Series(dtype=float)
            if not np.isfinite(a) or bvals.empty:
                continue
            b = float(bvals.mean())
            hv.append(float(a)); cv.append(b); diffs.append(float(a-b))
            pair_rows.append({"Day": day, "Metric": metric, "HitSymbol": hit_sym,
                              "HitValue": float(a), "MatchedControlMean": b, "Difference": float(a-b),
                              "ControlCount": len(bvals)})
        if not diffs:
            continue
        diffs = np.asarray(diffs)
        boot = rng.choice(diffs, size=(2000, len(diffs)), replace=True).mean(axis=1)
        metric_rows.append({"Day": day, "Metric": metric, "MatchedHits": len(diffs),
                            "HitMean": np.mean(hv), "MatchedControlMean": np.mean(cv),
                            "MeanDifference": np.mean(diffs),
                            "CI95Low": np.quantile(boot, .025), "CI95High": np.quantile(boot, .975),
                            "HitMedian": np.median(hv), "ControlMedian": np.median(cv)})
comparison = pd.DataFrame(metric_rows)
pair_metrics = pd.DataFrame(pair_rows)
comparison.to_csv(V2_DIR / "matched_landmark_comparison.csv", index=False)
pair_metrics.to_csv(V2_DIR / "matched_pair_metric_differences.csv", index=False)
display(comparison.loc[comparison.Metric.isin(["Return", "AboveSMA50", "AboveSMA200", "VolumeVsPre120", "Vol20VsPre120", "AboveTrough60High"])].round(3))


## 4. Recovery-event timing and charts

First observed close above 20/50/200-day SMA, first close above the **trough-date 60-day high**, first close at 2×/5×/10×, and first 20-day-high breakout. All event searches start *after* the trough and stop at 252 bars. An event's date is known only when that bar closes; a hypothetical trade would need next-bar execution. Never interpret first-breakout dates as known at the trough.

In [ ]:
def v2_first_events(dates, h, c, i, horizon=MIN_FOLLOWUP):
    last = min(i + horizon, len(c) - 1)
    ref_high = np.nanmax(h[i-59:i+1]) if i >= 59 else np.nan
    found = {"FirstAboveSMA20": np.nan, "FirstAboveSMA50": np.nan,
             "FirstAboveSMA200": np.nan, "FirstTrough60HighBreak": np.nan,
             "First20DayHighBreak": np.nan, "First2x": np.nan,
             "First5x": np.nan, "First10x": np.nan}
    for j in range(i+1, last+1):
        px = c[j]
        if not np.isfinite(px):
            continue
        for window, name in [(20, "FirstAboveSMA20"), (50, "FirstAboveSMA50"), (200, "FirstAboveSMA200")]:
            if np.isnan(found[name]) and j+1 >= window and px > np.nanmean(c[j-window+1:j+1]):
                found[name] = j-i
        if np.isnan(found["FirstTrough60HighBreak"]) and np.isfinite(ref_high) and px > ref_high:
            found["FirstTrough60HighBreak"] = j-i
        if np.isnan(found["First20DayHighBreak"]) and j >= 20 and px > np.nanmax(h[j-20:j]):
            found["First20DayHighBreak"] = j-i
        for multiple in (2, 5, 10):
            name = f"First{multiple}x"
            if np.isnan(found[name]) and px >= c[i]*multiple:
                found[name] = j-i
    return found

event_rows = []
for sym, g in panel.groupby("Symbol", sort=False, observed=True):
    sym = str(sym)
    if sym not in matched_hit_symbols and sym not in matched_control_symbols:
        continue
    dates, o, h, l, c, v = v2_arrays(g)
    for key in [key for key in cohort_keys if key[0] == sym]:
        rec = cohort_lookup[key]
        event_rows.append({"Symbol": sym, "Group": rec.Group, "TroughDate": key[1],
                           **v2_first_events(dates, h, c, int(rec.TroughIndex))})
events = pd.DataFrame(event_rows)
events.to_csv(V2_DIR / "recovery_event_timing.csv", index=False)
event_cols = [x for x in events if x.startswith("First")]
event_summary = events.groupby("Group")[event_cols].agg(["count", "median"])
display(event_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for group, label in [("hit_101x", "101x hits"), ("failed_new_low", "matched controls")]:
    x = forward[forward.Group.eq(group)]
    med = x.groupby("Day")["Return"].median()
    axes[0].plot(med.index, med.values, marker="o", label=label)
    medvol = x.groupby("Day")["Vol20VsPre120"].median()
    axes[1].plot(medvol.index, medvol.values, marker="o", label=label)
axes[0].set(title="Median return by observed landmark", xlabel="Trading days after trough", ylabel="Return")
axes[1].set(title="Median 20d volatility / pre-trough 120d", xlabel="Trading days after trough", ylabel="Ratio")
for ax in axes:
    ax.legend(); ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(V2_DIR / "post_trough_divergence.png", dpi=150)
plt.show(); plt.close(fig)


## 5. Diagnostics and limitations

- **Selection bias:** hit labels depend on future 101× achievement; this analysis describes historical differences, not prospective success rates.
- **Matched controls:** matched only on available OHLCV attributes, not market cap, fundamentals, liquidity or sector; matching can discard hits and cannot remove unobserved confounding.
- **Right censoring:** only observations with 252 subsequent bars are compared; non-hit status may change with later data.
- **Corporate actions:** validate price adjustment, split/bonus events, ticker changes, suspensions and delisted names before treating extreme multiples as economic returns.
- **Execution:** trough-close buying and close-confirmed events are descriptive; live entries require next-open execution and realistic costs.
- **Forward-data isolation:** do not feed 60/120/252-day features into earlier-landmark models.

The following cell checks uniqueness, match membership, horizon completeness and that event timings fall within the 252-bar window.

In [ ]:
assert not cohort.duplicated(["Symbol", "TroughDate"]).any(), "Duplicate cohort observations"
assert not matches.duplicated("ControlSymbol").any(), "A control was reused"
assert set(matches.HitSymbol).issubset(set(v2_hits.Symbol))
assert set(matches.ControlSymbol).issubset(set(v2_controls.Symbol))
assert set(forward.Day.unique()) == set(LANDMARKS)
assert forward.groupby(["Symbol", "TroughDate"]).Day.nunique().eq(len(LANDMARKS)).all()
assert forward.AsOfDate.ge(forward.TroughDate).all()
for col in event_cols:
    assert events[col].dropna().between(1, MIN_FOLLOWUP).all(), col
print("V2 structural checks passed")
print("Exports:")
for p in sorted(V2_DIR.iterdir()):
    print(" ", p.name)
